In [24]:
import gymnasium as gym
import time

In [ ]:
class MDP_FrozenLake:
    '''FrozenLake environment from Gymnasium.'''
    
    def __init__(self, policy=None):
        self.env = gym.make("FrozenLake-v1", 
                            desc=None, 
                            map_name="4x4", 
                            is_slippery=False, 
                            success_rate=1.0/3.0,
                            render_mode="human", 
                            reward_schedule=(10, -10, -1))

        self.total_reward = 0
        self.t = 0
        self.policy = policy if policy is not None else self.random
        
        self.observation = None
        self.episode_over = False
        
        self.history = []
        self.gamma = 0.95

    # S: State space
    def states(self):
        return range(self.env.observation_space.n)

    # A: Action space
    def actions(self):
        return {
            0: "LEFT",
            1: "DOWN",
            2: "RIGHT",
            3: "UP"}

    def discounted_return(self):
        G = 0
        for t, (_, _, reward, _) in enumerate(self.history):
            G += (self.gamma ** t) * reward
        return G

    # Utility functions
    def get_current_state(self, seed=None):
        self.observation, self.info = self.env.reset(seed=seed)
        print(f"Current observation: {self.observation},",
              f"Current info: {self.info}\n")

        self.episode_over = False
        self.total_reward = 0
        self.t = 0
        self.history = []

        return self.observation

    def take_action(self):
        state = self.observation

        action = self.policy(state) # Policy
        next_state, reward, terminated, truncated, info = self.env.step(action)

        self.observation = next_state # Update the state
        self.total_reward += reward # Assign a reward
        self.episode_over = terminated or truncated

        self.history.append((state, action, reward, next_state))

        print(f"t={self.t} | State={state} | "
              f"Action={self.actions()[action]} | "
              f"Next state={next_state} | Reward={reward}")

        time.sleep(0.5) 

        self.t += 1

        return next_state, reward

    # Policy: Random
    def random(self, state):
        return self.env.action_space.sample()

In [ ]:
# Run
def run_episode(mdp, seed=None):
    mdp.get_current_state(seed=seed)

    while not mdp.episode_over:
        state, reward = mdp.take_action()
        print(f"State: {state}, Reward: {reward}\n") 

    print("Total reward:", mdp.total_reward)
    print("Discounted return:", mdp.discounted_return())

    return mdp.history


mdp = MDP_FrozenLake()
print(list(mdp.states()))

mdp.policy = mdp.random
history = run_episode(mdp, seed=42)
mdp.env.close()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Current observation: 0, Current info: {'prob': 1}

t=0 | State=0 | Action=RIGHT | Next state=1 | Reward=-1
State: 1, Reward: -1

t=1 | State=1 | Action=UP | Next state=1 | Reward=-1
State: 1, Reward: -1

t=2 | State=1 | Action=RIGHT | Next state=2 | Reward=-1
State: 2, Reward: -1

t=3 | State=2 | Action=RIGHT | Next state=3 | Reward=-1
State: 3, Reward: -1

t=4 | State=3 | Action=LEFT | Next state=2 | Reward=-1
State: 2, Reward: -1

t=5 | State=2 | Action=RIGHT | Next state=3 | Reward=-1
State: 3, Reward: -1

t=6 | State=3 | Action=RIGHT | Next state=3 | Reward=-1
State: 3, Reward: -1

t=7 | State=3 | Action=RIGHT | Next state=3 | Reward=-1
State: 3, Reward: -1

t=8 | State=3 | Action=DOWN | Next state=7 | Reward=-10
State: 7, Reward: -10

Total reward: -18
Discounted return: -10.000000000000002
